# Daily Practice — 2026-11-19 — ML Testing & MLOps: Building a Data Contract Test Suite for an Incoming Data Pipeline

**Dataset:** [NYC Taxi Trips](https://github.com/mwaskom/seaborn-data) (6,433 real yellow/green cab
trips from March 2019, fetched live from seaborn's public dataset mirror) — pickup/dropoff times,
fare components, payment method, passenger count, and pickup/dropoff borough and zone.

## Problem statement

You're the QA engineer supporting a team that re-trains a taxi-fare model every night on a fresh
batch of trip records. Upstream data breaks silently far more often than a model's accuracy does:
a new payment processor starts sending an uncoded category, a GPS glitch logs a negative distance,
a zone-lookup table goes stale and nulls spike, a CSV export quietly turns a numeric column into
text for a handful of rows. None of this raises an exception. It just gets ingested, trains a
slightly worse model, and nobody notices until a dashboard looks off two weeks later.

Your job is to build a **data contract** — a declarative, per-column set of rules — and a
`validate_batch()` function that checks any incoming DataFrame against it and returns a
*structured* report, not a bare pass/fail. A QA engineer needs to know exactly **which** rule
broke, **how many rows** it affects, and **what the bad values looked like** — the same
information a good assertion failure gives you, applied to data instead of code.

## What you should produce

1. A `CONTRACT` dict covering every column that matters: dtype, nullability (with a maximum
   allowed null rate — real production data has *some* nulls, and a contract that demands zero
   is useless), numeric min/max bounds, and allowed categorical values where the domain is small
   and known.
2. `validate_batch(df, contract) -> list[dict]`, where each returned dict is one violation record
   with keys `column`, `rule`, `n_rows_affected`, `pct_rows_affected`, `example_values`.
3. Proof the contract is well-calibrated, not just strict-sounding: run it against **two**
   independently-sampled clean batches (zero violations on both — if your contract fails on
   clean data, it's too strict and will cry wolf in production) and against a **deliberately
   corrupted** batch containing five named, injected defects — your suite must catch every one
   of them, identified by the right column and rule.
4. A short write-up (TODO cell at the end) on which injected defects an aggregate check like
   "mean fare looks normal" would have missed, and which single defect would be most dangerous
   if it reached the fare-prediction model undetected.

*A note on the corrupted batch: the base data is real, not synthetic. The corruption injected in
Section 3 is a controlled test fixture — you can't wait around for production to hand you five
specific failure modes on demand, so we manufacture them here, on a holdout copy, purely to prove
the validator actually catches what it claims to catch.*

## Setup

In [ ]:
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 20)
RANDOM_STATE = 42

## Load the data and build reference / next-day / corrupted batches

We load the real taxi trips, shuffle once, and split into two disjoint batches: `reference_batch`
(what the contract will be designed against) and `next_day_batch` (an independent sample standing
in for "tomorrow's data" — if the contract only passes on the batch it was tuned on, it's
overfit). `inject_defects` is a **test fixture**, given to you — it manufactures a corrupted copy
of a batch with five specific, named defects so you have a known-bad case to validate your
checker against. You are not asked to write this function; you are asked to catch everything it
breaks.

In [ ]:
df = pd.read_csv(
    "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/taxis.csv"
)

shuffled = df.sample(frac=1.0, random_state=RANDOM_STATE).reset_index(drop=True)
half = len(shuffled) // 2
reference_batch = shuffled.iloc[:half].reset_index(drop=True)
next_day_batch = shuffled.iloc[half:].reset_index(drop=True)

print(reference_batch.shape, next_day_batch.shape)
reference_batch.head(3)

In [ ]:
def inject_defects(df_in, seed=0):
    '''Test fixture: returns (corrupted_copy, list_of_defect_names). Given, not a TODO.'''
    rng = np.random.RandomState(seed)
    df2 = df_in.copy()
    defects = []

    # 1. Unseen category: a payment processor code the contract has never seen
    idx = rng.choice(df2.index, size=max(1, int(0.02 * len(df2))), replace=False)
    df2.loc[idx, "payment"] = "mobile wallet"
    defects.append("unseen_category:payment")

    # 2. Out-of-range: a GPS glitch logs negative trip distance
    idx = rng.choice(df2.index, size=10, replace=False)
    df2.loc[idx, "distance"] = -(df2.loc[idx, "distance"] + 1)
    defects.append("out_of_range:distance (negative)")

    # 3. Out-of-range: a decimal-point/currency bug inflates a handful of fares
    idx = rng.choice(df2.index, size=5, replace=False)
    df2.loc[idx, "fare"] = 9999.0
    defects.append("out_of_range:fare (extreme high)")

    # 4. Null-rate spike: a stale zone-lookup join drops out for half the batch
    idx = rng.choice(df2.index, size=int(0.5 * len(df2)), replace=False)
    df2.loc[idx, "pickup_zone"] = np.nan
    defects.append("null_rate_spike:pickup_zone")

    # 5. Dtype violation: a CSV export turns a handful of passenger counts into text
    df2["passengers"] = df2["passengers"].astype(object)
    idx = rng.choice(df2.index, size=8, replace=False)
    df2.loc[idx, "passengers"] = "two"
    defects.append("dtype_violation:passengers")

    return df2, defects


corrupted_batch, injected_defects = inject_defects(next_day_batch, seed=1)
injected_defects

## Your task

Fill in the two TODOs below. `validate_batch`'s signature and return-value contract are fixed —
the solution cells and the write-up both assume a list of dicts with exactly those five keys.

Start `CONTRACT` from the two columns given as worked examples, then cover the rest:
`distance`, `fare`, `tip`, `tolls`, `total` (all numeric), `color`, `payment`, `pickup_borough`,
`dropoff_borough` (categorical with a known, small domain), and `pickup_zone`, `dropoff_zone`
(categorical, but with ~200 distinct values each — too many to enumerate, so only check
nullability for those two).

In [ ]:
CONTRACT = {
    "passengers": {"dtype": "numeric", "nullable": False, "min": 0, "max": 6},
    "color": {"dtype": "categorical", "nullable": False, "allowed_values": {"yellow", "green"}},

    # TODO: "distance" -- numeric, not nullable, real trips run roughly 0 to ~37 miles;
    #       leave headroom but a negative distance should never pass.
    # TODO: "fare", "tip", "tolls", "total" -- numeric, not nullable, each non-negative with
    #       a sensible upper bound (look at reference_batch.describe() for realistic scale).
    # TODO: "payment" -- categorical, nullable (some trips never get a payment code logged --
    #       check the real null rate in reference_batch first and set max_null_rate above it),
    #       allowed_values is {"credit card", "cash"}.
    # TODO: "pickup_borough", "dropoff_borough" -- categorical, nullable, allowed_values is the
    #       five NYC boroughs.
    # TODO: "pickup_zone", "dropoff_zone" -- categorical, nullable, no allowed_values (too many
    #       distinct zones to enumerate) -- just a max_null_rate.
}

In [ ]:
def validate_batch(df_in, contract):
    '''
    Check df_in against contract. Returns a list of violation dicts, each with keys:
      column, rule, n_rows_affected, pct_rows_affected, example_values

    For every column in `contract`:
      1. Missing column -> one violation, rule="missing_column".
      2. Null rate above the allowed max (0.0 if not nullable, else rule["max_null_rate"])
         -> one violation, rule="null_rate_exceeded(...)".
      3. For dtype == "numeric": values that don't coerce to numeric -> "dtype_violation(...)";
         among the ones that do, values below "min" -> "below_min(...)", above "max" ->
         "above_max(...)". (Hint: pd.to_numeric(series, errors="coerce") turns bad values
         into NaN without raising.)
      4. For dtype == "categorical" with "allowed_values" given: non-null values outside that
         set -> "unexpected_category".
    Only emit a violation dict when at least one row actually trips that check.
    '''
    # TODO: implement
    violations = []
    return violations

In [ ]:
# TODO: run validate_batch on all three batches and print the results.
# Expect: 0 violations on reference_batch, 0 on next_day_batch, 5 on corrupted_batch
# (one per injected defect, naming the right column each time).

print("--- reference_batch ---")
# TODO

print("--- next_day_batch ---")
# TODO

print("--- corrupted_batch ---")
# TODO

### Write-up

TODO: in 3–5 sentences — for each of the five injected defects, would an aggregate sanity check
like "mean fare looks about right" or "row count looks about right" have caught it? Which single
defect would do the most damage to the fare-prediction model if it slipped through *silently*
(no exception, no validator), and why that one specifically rather than the others?

*(your answer here)*

---

## Solution

*(scroll down when you're ready — try it yourself first)*

<details>
<summary>Click to reveal solution</summary>

```python
CONTRACT = {
    "passengers": {"dtype": "numeric", "nullable": False, "min": 0, "max": 6},
    "color": {"dtype": "categorical", "nullable": False, "allowed_values": {"yellow", "green"}},
    "distance": {"dtype": "numeric", "nullable": False, "min": 0, "max": 100},
    "fare": {"dtype": "numeric", "nullable": False, "min": 0, "max": 300},
    "tip": {"dtype": "numeric", "nullable": False, "min": 0, "max": 100},
    "tolls": {"dtype": "numeric", "nullable": False, "min": 0, "max": 60},
    "total": {"dtype": "numeric", "nullable": False, "min": 0, "max": 400},
    "payment": {
        "dtype": "categorical", "nullable": True, "max_null_rate": 0.02,
        "allowed_values": {"credit card", "cash"},
    },
    "pickup_borough": {
        "dtype": "categorical", "nullable": True, "max_null_rate": 0.02,
        "allowed_values": {"Manhattan", "Brooklyn", "Queens", "Bronx", "Staten Island"},
    },
    "dropoff_borough": {
        "dtype": "categorical", "nullable": True, "max_null_rate": 0.02,
        "allowed_values": {"Manhattan", "Brooklyn", "Queens", "Bronx", "Staten Island"},
    },
    "pickup_zone": {"dtype": "categorical", "nullable": True, "max_null_rate": 0.02},
    "dropoff_zone": {"dtype": "categorical", "nullable": True, "max_null_rate": 0.02},
}


def validate_batch(df_in, contract):
    violations = []
    for col, rule in contract.items():
        if col not in df_in.columns:
            violations.append({
                "column": col, "rule": "missing_column",
                "n_rows_affected": len(df_in), "pct_rows_affected": 100.0,
                "example_values": [],
            })
            continue

        series = df_in[col]
        null_mask = series.isna()
        null_rate = null_mask.mean()
        max_null_rate = rule.get("max_null_rate", 0.0 if not rule.get("nullable", False) else 1.0)
        if null_rate > max_null_rate:
            violations.append({
                "column": col, "rule": f"null_rate_exceeded(>{max_null_rate:.1%})",
                "n_rows_affected": int(null_mask.sum()),
                "pct_rows_affected": round(float(null_rate) * 100, 2),
                "example_values": [],
            })

        non_null = series[~null_mask]

        if rule["dtype"] == "numeric":
            coerced = pd.to_numeric(non_null, errors="coerce")
            bad_dtype_mask = coerced.isna()
            if bad_dtype_mask.any():
                violations.append({
                    "column": col, "rule": "dtype_violation(non_numeric)",
                    "n_rows_affected": int(bad_dtype_mask.sum()),
                    "pct_rows_affected": round(float(bad_dtype_mask.mean()) * 100, 2),
                    "example_values": non_null[bad_dtype_mask].unique()[:3].tolist(),
                })
            numeric_vals = coerced[~bad_dtype_mask]
            if "min" in rule:
                below = numeric_vals < rule["min"]
                if below.any():
                    violations.append({
                        "column": col, "rule": f"below_min({rule['min']})",
                        "n_rows_affected": int(below.sum()),
                        "pct_rows_affected": round(float(below.mean()) * 100, 2),
                        "example_values": numeric_vals[below].unique()[:3].tolist(),
                    })
            if "max" in rule:
                above = numeric_vals > rule["max"]
                if above.any():
                    violations.append({
                        "column": col, "rule": f"above_max({rule['max']})",
                        "n_rows_affected": int(above.sum()),
                        "pct_rows_affected": round(float(above.mean()) * 100, 2),
                        "example_values": numeric_vals[above].unique()[:3].tolist(),
                    })

        elif rule["dtype"] == "categorical" and "allowed_values" in rule:
            bad_cat_mask = ~non_null.isin(rule["allowed_values"])
            if bad_cat_mask.any():
                violations.append({
                    "column": col, "rule": "unexpected_category",
                    "n_rows_affected": int(bad_cat_mask.sum()),
                    "pct_rows_affected": round(float(bad_cat_mask.mean()) * 100, 2),
                    "example_values": non_null[bad_cat_mask].unique()[:3].tolist(),
                })

    return violations


for name, batch in [("reference_batch", reference_batch),
                     ("next_day_batch", next_day_batch),
                     ("corrupted_batch", corrupted_batch)]:
    v = validate_batch(batch, CONTRACT)
    print(f"--- {name}: {len(v)} violation(s) ---")
    for rec in v:
        print(rec)
```

Running this gives **0 violations** on both `reference_batch` and `next_day_batch` — the contract
is calibrated against real variation, not just the one sample it was written next to — and
**exactly 5 violations** on `corrupted_batch`, one per injected defect, each pointing at the right
column: `passengers` / dtype_violation, `distance` / below_min, `fare` / above_max, `payment` /
unexpected_category, `pickup_zone` / null_rate_exceeded.

**Write-up:**
An aggregate check on `fare` (e.g. "mean fare within historical range") would likely *miss* the
extreme-fare defect entirely — it only hits 5 rows out of ~3,200, nowhere near enough to move a
mean, let alone a median. The same is true of the negative-distance and unseen-payment-category
defects: each touches well under 2% of rows, so almost any batch-level aggregate (mean, median,
total row count) stays well inside "normal," and a QA process that only watches aggregates
ships all three straight to the next training run. Only the null-rate spike (50% of `pickup_zone`
suddenly missing) is large enough that a naive "did row count change" or "did null count change"
check might stumble onto it by accident — and only if someone is actually watching that
column specifically.

The most dangerous defect if it slipped through silently is the **unseen payment category**
(`"mobile wallet"`). Numeric defects (negative distance, extreme fare) are jarring enough that a
downstream feature-engineering step or a naive outlier filter has a decent chance of flagging or
clipping them even without an explicit contract. A brand-new string value in a categorical column
that gets one-hot encoded, by contrast, fails *quietly and structurally*: it either produces a
column the trained model has never seen at serving time (depending on the encoder, this silently
becomes an all-zero row or crashes production, neither of which is caught by a metric dashboard
until well after the bad model is live) or gets silently bucketed into "unknown" and biases
every prediction for mobile-wallet riders the same way. It's the one defect that has nothing to
do with value *magnitude*, so magnitude-based anomaly detection — the thing most teams reach for
first — won't catch it at all.

</details>